# L4 — RAG (Retrieval-Augmented Generation)

> **El modelo responde basándose en documentos que tú le proporcionas en el contexto, no en lo que aprendió durante el entrenamiento.**

## ¿Qué vas a aprender?

- Qué es un **embedding** y por qué permite buscar por significado
- Qué es una **base de datos vectorial** y por qué la necesitamos
- Las **dos fases** de RAG: indexación (offline) y consulta (online)
- Cómo construir un sistema RAG completo con ChromaDB + Claude

## ¿Qué es RAG?

Un LLM sabe mucho — pero **solo lo que aprendió durante su entrenamiento**. No sabe nada de tu documentación interna, de los tickets de tu empresa, del estado de tu sistema hoy, ni de nada que haya ocurrido después de su fecha de corte.

**RAG resuelve eso:** en lugar de pedirle al modelo que "recuerde" información que no tiene, se la **buscamos nosotros** y se la metemos en el contexto justo antes de que responda.

```
Sin RAG:  Pregunta → LLM → Respuesta (basada solo en entrenamiento)
Con RAG:  Pregunta → Buscar documentos → Pregunta + Contexto → LLM → Respuesta fundada
```

## Los dos conceptos nuevos

### Embedding

Un embedding es una **representación numérica del significado** de un texto: una lista de números (un vector) donde cada número captura algún aspecto semántico.

```
"El servicio de login está caído"  → [0.23, -0.87, 0.41, 0.12, ...]   (384 números)
"Users cannot authenticate"        → [0.24, -0.85, 0.39, 0.11, ...]   (muy similar)
"El tiempo en Zaragoza mañana"     → [0.91,  0.12, -0.67, 0.88, ...]  (muy diferente)
```

**La propiedad clave:** textos con significado similar producen vectores similares, **aunque usen palabras completamente distintas**. Esto permite buscar por significado en lugar de por palabras exactas.

### Base de datos vectorial

Una BD diseñada para almacenar vectores y buscar los más similares a uno dado de forma eficiente.

| SQL tradicional | Base de datos vectorial |
|-----|------------------------|
| `WHERE texto LIKE '%login%'` | "Dame los chunks más similares a esta pregunta" |
| Busca palabras exactas | Busca significado semántico |
| No entiende sinónimos | Entiende sinónimos y paráfrasis |

Aquí usamos **Chroma** porque corre en local sin configuración. En producción se suele usar **pgvector** (extensión de PostgreSQL).

## Las dos fases de RAG

### Fase 1 — Indexación (se hace una vez o periódicamente)

```
Documentos
    ↓
Dividir en chunks (~500 palabras con solapamiento)
    ↓
Calcular el embedding de cada chunk
    ↓
Guardar chunk + embedding en la BD vectorial
```

### Fase 2 — Consulta (se hace en cada pregunta)

```
Pregunta del usuario
    ↓
Calcular el embedding de la pregunta (mismo modelo)
    ↓
Buscar los N chunks más similares en la BD
    ↓
Construir el prompt: pregunta + chunks como contexto
    ↓
LLM genera la respuesta basándose en el contexto
```

> **Crítico:** el modelo de embeddings tiene que ser el **mismo** en indexación y en consulta. Si cambias el modelo, hay que reindexar desde cero.

## Conceptos adicionales

### Chunking strategy
Las estrategias más comunes:
- **Fixed size** (lo que usamos aquí): chunks de N caracteres con solapamiento. Simple, funciona bien.
- **Recursive**: divide por párrafos, luego frases, luego palabras hasta llegar al tamaño objetivo.
- **Semantic**: usa embeddings para dividir donde cambia el tema. Más caro pero más preciso.

### Top-K retrieval
Cuántos chunks recuperar. Demasiado bajo → falta contexto. Demasiado alto → ruido y más tokens. **K=3 o K=5** es un buen punto de partida.

### Hallucination vs grounding
Sin RAG el modelo puede "alucinar" información plausible pero incorrecta. Con RAG el modelo tiene el documento real en el contexto, lo que reduce drásticamente las alucinaciones — siempre que el retrieval haya encontrado los chunks correctos.

> **Requisitos:** `pip install anthropic chromadb sentence-transformers python-dotenv`

---
# PARTE 1 — Indexación (`indexer.py`)

> **Ejecuta esta parte UNA VEZ** (o cada vez que cambien los documentos). Construye la base de datos vectorial en `./chroma_db/`.

Lo que vamos a hacer:
1. Leer todos los `.md` de `./docs/`
2. Dividir cada documento en chunks con solapamiento
3. Calcular el embedding de cada chunk
4. Guardar chunk + embedding + metadatos en ChromaDB

## Setup de la indexación

Usamos `sentence-transformers` con el modelo **`all-MiniLM-L6-v2`** (~90 MB, corre en CPU sin problemas). Se descarga automáticamente la primera vez.

**Parámetros que vamos a usar:**
- `CHUNK_SIZE = 500` caracteres por chunk
- `CHUNK_OVERLAP = 100` caracteres de solapamiento entre chunks consecutivos

In [ ]:
"""
L4 — RAG Indexer: divide documentos en chunks y los guarda en Chroma
====================================================================
Primera fase de RAG: la indexación.
Se ejecuta una vez (o cada vez que cambian los documentos).

Qué hace este script:
    1. Lee todos los .md de la carpeta docs/
    2. Divide cada documento en chunks con solapamiento
    3. Calcula el embedding de cada chunk (modelo local, sin API key)
    4. Guarda chunk + embedding + metadatos en ChromaDB

El resultado queda en ./chroma_db/ — una base de datos vectorial persistente
que retriever.py y rag_pipeline.py consultarán después.

Requisitos:
    pip install chromadb sentence-transformers

    No necesita API key — los embeddings se calculan en local con
    sentence-transformers. En el primer run descarga el modelo (~90 MB).
"""

import os
import glob
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
# ─────────────────────────────────────────────
# Configuración
# ─────────────────────────────────────────────

DOCS_PATH       = "./docs"
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "support_docs"

# all-MiniLM-L6-v2: modelo pequeño (90 MB), rápido, buena calidad para búsqueda semántica.
# Este mismo modelo HAY QUE USARLO también en retriever.py — si indexas con A y
# buscas con B, los vectores son incompatibles y los resultados serán basura.
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

CHUNK_SIZE    = 500   # caracteres por chunk
CHUNK_OVERLAP = 100   # caracteres de solapamiento entre chunks consecutivos

## Función de chunking

**Por qué solapamiento:** si una idea importante cae justo en el límite entre dos chunks, el solapamiento garantiza que aparece completa en al menos uno y no queda cortada.

```
texto:  "abcdefghijklmnopqrst"  (chunk_size=10, overlap=3)
chunk1: "abcdefghij"             (start=0)
chunk2: "hijklmnopq"             (start=7)  ← 'hij' se repite
chunk3: "nopqrst"                (start=14)
```

In [ ]:
# ─────────────────────────────────────────────
# Chunking
# ─────────────────────────────────────────────

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """
    Divide el texto en chunks de tamaño fijo con solapamiento.

    Por qué solapamiento: si una idea importante cae justo en el límite
    entre dos chunks, el solapamiento garantiza que aparece completa
    en al menos uno de ellos y no queda cortada.

    Ejemplo con chunk_size=10, overlap=3:
        texto:  "abcdefghijklmnopqrst"
        chunk1: "abcdefghij"           (start=0)
        chunk2: "hijklmnopq"           (start=7)  ← 'hij' se repite
        chunk3: "nopqrst"              (start=14)
    """
    chunks = []
    start  = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        # Avanzamos (chunk_size - overlap) para que el siguiente chunk
        # empiece overlap caracteres antes del final del actual
        start += chunk_size - overlap
    return chunks

## Carga de documentos

Lee todos los `.md` de `./docs/` y guarda su contenido + el nombre del fichero como metadato (para poder citar la fuente más tarde — esto se llama *source attribution*).

In [ ]:
# ─────────────────────────────────────────────
# Carga de documentos
# ─────────────────────────────────────────────

def load_documents(docs_path: str) -> list[dict]:
    """
    Lee todos los archivos .md de la carpeta y devuelve una lista de dicts
    con el contenido y el nombre del fichero como fuente.

    Guardamos el nombre del fichero como metadato para poder decirle al
    usuario de dónde viene cada respuesta — esto es lo que llaman "citations"
    o "source attribution" en sistemas RAG de producción.
    """
    documents = []
    pattern   = os.path.join(docs_path, "*.md")

    for filepath in sorted(glob.glob(pattern)):
        filename = os.path.basename(filepath)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        documents.append({"source": filename, "content": content})
        print(f"  Cargado: {filename} ({len(content)} caracteres)")

    return documents

## Indexación: chunkear + embedder + guardar

Por cada documento:
1. Lo dividimos en chunks
2. Para cada chunk: lo añadimos a Chroma con un id único + metadatos
3. Chroma calcula el embedding internamente usando `embedding_fn`

In [ ]:
# ─────────────────────────────────────────────
# Indexación
# ─────────────────────────────────────────────

def index_documents(documents: list[dict], collection) -> int:
    """
    Divide cada documento en chunks y los añade a la colección de Chroma.

    Cada chunk se almacena con:
        - document: el texto del chunk (lo que se buscará)
        - embedding: el vector calculado por embedding_fn (Chroma lo hace automáticamente)
        - metadata: fuente y número de chunk (para atribución y depuración)
        - id: identificador único — Chroma lo requiere para evitar duplicados
    """
    total_chunks = 0

    for doc in documents:
        chunks = chunk_text(doc["content"])
        source = doc["source"]

        print(f"\n  {source}: {len(chunks)} chunks")

        ids       = []
        texts     = []
        metadatas = []

        for i, chunk in enumerate(chunks):
            chunk_id = f"{source}__chunk_{i}"
            ids.append(chunk_id)
            texts.append(chunk)
            metadatas.append({"source": source, "chunk_index": i})

            # Mostramos los primeros 60 chars para ver qué se indexa
            preview = chunk[:60].replace("\n", " ")
            print(f"    [{i}] {preview}...")

        # Chroma llama a embedding_fn internamente para cada texto
        # y almacena el resultado junto al texto y los metadatos
        collection.add(documents=texts, metadatas=metadatas, ids=ids)
        total_chunks += len(chunks)

    return total_chunks

## Función `main` — orquesta toda la indexación

Crea (o recrea) la colección `support_docs` y la rellena con todos los chunks. Usa **distancia coseno** como métrica de similitud (la más común para embeddings de texto).

In [ ]:
# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main():
    print("=" * 60)
    print("FASE 1 — INDEXACIÓN")
    print("=" * 60)

    # PersistentClient guarda los datos en disco.
    # La próxima vez que lo abras (retriever.py, rag_pipeline.py)
    # los embeddings ya están calculados — no hay que recalcularlos.
    client = chromadb.PersistentClient(path=CHROMA_PATH)

    embedding_fn = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

    # get_or_create_collection: si ya existe (de una indexación anterior), la borramos
    # y recreamos para que los cambios en los docs se reflejen.
    # En producción se usaría una estrategia de actualización incremental.
    try:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"\nColección anterior eliminada — reindexando desde cero.")
    except Exception:
        pass  # no existía, no hay problema

    collection = client.create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        # cosine: mide similitud de dirección entre vectores (independiente de la magnitud)
        # Es la métrica más usada para comparar embeddings de texto
        metadata={"hnsw:space": "cosine"},
    )

    print(f"\nCargando documentos desde '{DOCS_PATH}'...")
    documents = load_documents(DOCS_PATH)

    if not documents:
        print(f"\nERROR: No se encontraron archivos .md en '{DOCS_PATH}'")
        return

    print(f"\nIndexando {len(documents)} documentos...")
    total = index_documents(documents, collection)

    print(f"\n{'=' * 60}")
    print(f"Indexación completada: {total} chunks en '{CHROMA_PATH}'")
    print(f"Modelo de embeddings: {EMBEDDING_MODEL}")
    print(f"Chunk size: {CHUNK_SIZE} chars | Overlap: {CHUNK_OVERLAP} chars")
    print(f"{'=' * 60}")
    print("\nAhora puedes ejecutar:")
    print("  python retriever.py         ← probar el retrieval")
    print("  python rag_pipeline.py      ← RAG completo con LLM")

## Ejecutar la indexación

Lanza esto **una vez**. Verás la lista de documentos cargados, los chunks generados y el resumen final:

In [ ]:
main()

---
# PARTE 2 — Retrieval (`retriever.py`)

> **Lo que vamos a probar:** dado una pregunta, recuperar los chunks más relevantes de la BD vectorial. Sin LLM todavía — solo retrieval puro.

Esto es muy útil para **depurar**: si los chunks recuperados no son relevantes, el LLM no podrá responder bien — hay que arreglar el retrieval primero.

## Setup del retriever

**CRÍTICO:** mismo modelo de embeddings que en la indexación. Si cambias el modelo aquí sin reindexar, los vectores de la query y los almacenados viven en espacios diferentes y la búsqueda no tiene sentido.

In [ ]:
"""
L4 — RAG Retriever: busca los chunks más relevantes para una pregunta
=====================================================================
Segunda pieza del sistema RAG. Puede usarse de dos formas:

    1. Como script independiente (para probar que el retrieval funciona
       antes de conectar el LLM — muy útil para depurar).

    2. Importado por rag_pipeline.py:
       from retriever import retrieve

La función clave: `retrieve(query, top_k)` → lista de chunks ordenados
por relevancia semántica.

Requisito previo: ejecutar indexer.py al menos una vez.

Requisitos:
    pip install chromadb sentence-transformers
"""

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
# ─────────────────────────────────────────────
# Configuración — debe coincidir con indexer.py
# ─────────────────────────────────────────────

CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "support_docs"

# CRÍTICO: usar exactamente el mismo modelo que se usó al indexar.
# Si cambias el modelo aquí sin reindexar, los vectores de la query
# y los vectores almacenados viven en espacios diferentes y la
# búsqueda no tiene sentido semántico.
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

TOP_K_DEFAULT = 3  # cuántos chunks recuperar por defecto

## Conexión lazy a la colección

Una variable global que se inicializa al primer uso. Evita abrir la conexión múltiples veces si llamamos a `retrieve()` en un bucle.

In [ ]:
# ─────────────────────────────────────────────
# Conexión a la colección (lazy, al primer uso)
# ─────────────────────────────────────────────

_collection = None

def _get_collection():
    """
    Abre la conexión a Chroma la primera vez que se necesita.
    Usar una variable global evita abrir la conexión múltiples
    veces si retrieve() se llama en un bucle.
    """
    global _collection
    if _collection is None:
        client = chromadb.PersistentClient(path=CHROMA_PATH)
        embedding_fn = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
        _collection = client.get_collection(
            name=COLLECTION_NAME,
            embedding_function=embedding_fn,
        )
    return _collection

## La función `retrieve` — el corazón del sistema

Proceso interno:
1. Chroma calcula el embedding de la query con el mismo modelo
2. Compara contra todos los vectores almacenados usando similitud coseno
3. Devuelve los `top_k` chunks más similares

Convertimos la **distancia coseno** `[0, 2]` a una **similitud** `[0, 1]` más intuitiva:
- distancia=0 → similitud=1.0 (idéntico)
- distancia=2 → similitud=0.0 (opuesto)

In [ ]:
# ─────────────────────────────────────────────
# Función principal de retrieval
# ─────────────────────────────────────────────

def retrieve(query: str, top_k: int = TOP_K_DEFAULT) -> list[dict]:
    """
    Busca los chunks más relevantes para la pregunta dada.

    Proceso interno:
        1. Chroma calcula el embedding de `query` usando el mismo modelo
           que usamos al indexar (embedding_fn).
        2. Compara ese vector contra todos los vectores almacenados
           usando similitud coseno.
        3. Devuelve los top_k chunks con mayor similitud.

    Devuelve una lista de dicts con:
        - text:       el texto del chunk
        - source:     nombre del documento de origen
        - similarity: puntuación [0.0, 1.0] — cuánto se parece a la query
        - chunk_index: posición del chunk dentro del documento
    """
    collection = _get_collection()

    results = collection.query(
        query_texts=[query],
        n_results=top_k,
    )

    # results["documents"][0]  → lista de textos (la [0] es porque query_texts acepta batch)
    # results["metadatas"][0]  → lista de metadatos
    # results["distances"][0]  → lista de distancias coseno [0, 2]
    #                            0 = idéntico, 2 = opuesto

    chunks = []
    docs      = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for text, meta, distance in zip(docs, metadatas, distances):
        # Convertimos distancia coseno a similitud [0, 1]
        # distancia=0 → similitud=1.0 (idéntico)
        # distancia=2 → similitud=0.0 (opuesto)
        similarity = 1.0 - (distance / 2.0)

        chunks.append({
            "text":        text,
            "source":      meta["source"],
            "chunk_index": meta["chunk_index"],
            "similarity":  round(similarity, 3),
        })

    return chunks

## Probar el retrieval

Las queries de prueba cubren preguntas sobre los tres documentos. Si los chunks recuperados son relevantes, el retrieval funciona y podemos pasar al LLM.

In [ ]:
# ─────────────────────────────────────────────
# Uso como script (para probar el retrieval)
# ─────────────────────────────────────────────

def print_results(query: str, chunks: list[dict]) -> None:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    for i, chunk in enumerate(chunks, 1):
        print(f"\n[{i}] {chunk['source']} (chunk {chunk['chunk_index']}) — similitud: {chunk['similarity']:.1%}")
        print(chunk["text"][:300].replace("\n", " ") + "...")
    print()


# Preguntas de prueba para verificar que el retrieval funciona correctamente
# antes de conectar el LLM. Si los chunks recuperados no son relevantes,
# el LLM no va a poder responder bien — hay que arreglar el retrieval primero.
TEST_QUERIES = [
    "¿Qué hago cuando los usuarios no pueden hacer login?",
    "El servicio está devolviendo errores 500 en autenticación",
    "¿Cómo soluciono un problema de conexiones a la base de datos?",
    "Los clientes están recibiendo errores 429",
    "¿Qué pasa cuando el circuit breaker se abre?",
]

Ejecuta el bucle de pruebas y observa qué chunks se recuperan para cada query:

In [ ]:
for query in TEST_QUERIES:
    chunks = retrieve(query)
    print_results(query, chunks)

---
# PARTE 3 — Pipeline completo (`rag_pipeline.py`)

> **Por fin unimos todo:** retrieval + LLM. El modelo responde basándose **exclusivamente** en los chunks recuperados.

## Setup del pipeline

Importamos la función `retrieve` que acabamos de definir. Usamos **Claude Haiku** porque es rápido y barato — en RAG el contexto lo provee el retrieval, no el modelo, así que no necesitamos el modelo más potente.

In [ ]:
"""
L4 — RAG Pipeline: recupera contexto y genera la respuesta con el LLM
======================================================================
Tercera y última pieza: une retrieval + generación.

Flujo completo de cada pregunta:
    Pregunta
        ↓
    retrieve()  →  chunks relevantes de la documentación
        ↓
    build_prompt()  →  prompt con contexto inyectado
        ↓
    Claude  →  respuesta fundamentada en los documentos reales
        ↓
    Respuesta + fuentes citadas

La diferencia clave con L2/L3: el LLM no responde desde su entrenamiento.
Responde desde los documentos que le pasamos en el contexto.
Si el retrieval falla, el modelo lo admite — no alucina una respuesta.

Requisitos:
    pip install anthropic chromadb sentence-transformers
    export ANTHROPIC_API_KEY="sk-ant-..."

Requisito previo: ejecutar indexer.py al menos una vez.
"""

from pathlib import Path
from dotenv import load_dotenv
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / ".env").exists():
        load_dotenv(_p / ".env", override=True)
        break
import json
import anthropic
# from retriever import retrieve  # ← retrieve ya está en el namespace del notebook

client = anthropic.Anthropic()
MODEL  = "claude-haiku-4-5-20251001"  # Haiku: rápido y barato para RAG. En RAG el
                                       # contexto lo provee el retrieval, no el modelo.

## Construcción del prompt con contexto

El formato importa: **separar claramente** los chunks del contexto y la pregunta ayuda al modelo a distinguir entre "lo que debe usar" y "lo que debe responder".

Cada chunk lleva su fuente para que el modelo pueda citarla en la respuesta.

In [ ]:
# ─────────────────────────────────────────────
# Construcción del prompt con contexto
# ─────────────────────────────────────────────

SYSTEM_PROMPT = """
Eres un agente de soporte técnico especializado. Responde preguntas basándote
EXCLUSIVAMENTE en la documentación que se te proporciona en el contexto.

Reglas:
- Si la respuesta está en el contexto, respóndela citando la fuente entre corchetes: [nombre_doc.md]
- Si el contexto no contiene información suficiente para responder, dilo claramente:
  "La documentación disponible no cubre este caso."
- No uses tu conocimiento de entrenamiento para completar lo que falta en el contexto.
- Responde en español, de forma concisa y accionable.
""".strip()


def build_prompt(question: str, chunks: list[dict]) -> str:
    """
    Construye el mensaje de usuario con la pregunta y los chunks como contexto.

    El formato importa: separar claramente los chunks del contexto y la pregunta
    ayuda al modelo a distinguir entre "lo que debe usar" y "lo que debe responder".
    Cada chunk lleva su fuente para que el modelo pueda citarla.
    """
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"--- Fragmento {i} de {chunk['source']} ---\n{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    return f"""DOCUMENTACIÓN RELEVANTE:

{context}

---

PREGUNTA:
{question}"""

## La función `ask` — pipeline RAG completo

Tres pasos:
1. **Retrieve** — recuperar los chunks más relevantes
2. **Build prompt** — inyectar los chunks como contexto
3. **Generate** — pedir al LLM que responda basándose **exclusivamente** en ese contexto

El system prompt instruye al modelo a citar fuentes y a admitir cuando no tenga información — **eso reduce las alucinaciones**.

In [ ]:
# ─────────────────────────────────────────────
# Pipeline principal
# ─────────────────────────────────────────────

def ask(question: str, top_k: int = 3, verbose: bool = True) -> dict:
    """
    Responde una pregunta usando RAG.

    Parámetros:
        question: la pregunta del usuario
        top_k:    cuántos chunks recuperar (más chunks = más contexto pero más tokens)
        verbose:  si True, muestra el proceso paso a paso

    Devuelve un dict con la respuesta y los metadatos del retrieval.
    """
    if verbose:
        print(f"\n{'=' * 60}")
        print(f"PREGUNTA: {question}")
        print(f"{'=' * 60}")

    # Paso 1: recuperar chunks relevantes
    chunks = retrieve(question, top_k=top_k)

    if verbose:
        print(f"\n[Retrieval] {len(chunks)} chunks recuperados:")
        for chunk in chunks:
            print(f"  • {chunk['source']} (chunk {chunk['chunk_index']}) — {chunk['similarity']:.1%}")

    # Paso 2: construir el prompt con el contexto inyectado
    user_message = build_prompt(question, chunks)

    # Paso 3: llamar al LLM con el contexto como parte del prompt
    # El modelo solo "ve" los chunks recuperados, no la documentación completa.
    # Esto es más eficiente (menos tokens) y más preciso (solo lo relevante).
    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0.1,  # casi determinista — es una pregunta con respuesta en el contexto
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )

    answer = response.content[0].text.strip()

    if verbose:
        print(f"\n[Respuesta]")
        print(answer)
        print(f"\n[Fuentes usadas]")
        sources = list({chunk["source"] for chunk in chunks})
        for src in sources:
            print(f"  • {src}")

    return {
        "question": question,
        "answer":   answer,
        "chunks":   chunks,
        "sources":  list({chunk["source"] for chunk in chunks}),
        "model":    MODEL,
    }

## Preguntas de ejemplo

Cubrimos varios casos:
- Preguntas con respuesta directa en un documento
- Una pregunta que requiere combinar información de **dos** documentos
- Una pregunta **fuera de cobertura** — el modelo debe admitir que no tiene la información (no inventar)

In [ ]:
# ─────────────────────────────────────────────
# Ejemplos de uso
# ─────────────────────────────────────────────

# Preguntas diseñadas para cubrir los tres documentos y casos edge:
# - Preguntas directas (respuesta clara en la doc)
# - Pregunta fuera de cobertura (el modelo debe admitir que no sabe)

EXAMPLE_QUESTIONS = [
    # Documentado en authentication.md
    "¿Qué hago cuando el servicio de login devuelve errores 500?",

    # Documentado en database.md
    "Hay demasiadas conexiones a la base de datos y los servicios fallan. ¿Cómo lo soluciono?",

    # Documentado en api_gateway.md
    "Un cliente enterprise está recibiendo errores 429. ¿Por qué puede pasar eso?",

    # Requiere combinar auth + db
    "Los usuarios no pueden autenticarse y veo timeouts en los logs. ¿Por dónde empiezo?",

    # Fuera de la documentación disponible — el modelo debe decir que no sabe
    "¿Cómo configuro alertas de Prometheus para el servicio de pagos?",
]

## Ejecutar el pipeline RAG

Lanza todas las preguntas. Observa para cada una:
- Qué chunks se recuperan
- Si el modelo cita correctamente las fuentes
- Cómo responde a la pregunta fuera de cobertura

In [ ]:
for question in EXAMPLE_QUESTIONS:
    ask(question)
    print()

## Cómo sería esto en producción

Aquí lo construimos como tres scripts que corren en tu máquina. En producción:

| Componente | En este notebook | En producción |
|---|---|---|
| BD vectorial | ChromaDB (fichero local) | pgvector (PostgreSQL) o Qdrant |
| Modelo de embeddings | Local via sentence-transformers | Servicio dedicado (CPU/GPU) |
| LLM | Claude API | Claude API (igual) |
| Cola de indexación | No existe | SQS, Celery, o eventos |
| Caché | No existe | Redis (preguntas frecuentes) |

La indexación se vuelve **event-driven** o **batch periódico** (cron nocturno), y la consulta corre como un endpoint HTTP detrás de tu API.

## Siguientes pasos → L5

Con RAG el sistema puede responder sobre documentación estática. El siguiente nivel son los **agentes**: sistemas que combinan tool use, RAG y razonamiento autónomo para perseguir **objetivos complejos**, decidiendo por sí mismos qué pasos dar y en qué orden.

→ **[L5 — Agente](../L5-AGENT/L5_Agente.ipynb)**